In [0]:
# 00_fetch_live_oem.ipynb
import os
import csv
import requests
from datetime import datetime, timezone, timedelta

MISSION_START_UTC = "2026-04-02 02:00"
STEP_SIZE = "5 m"
TARGET_COMMAND = "-1024"
CENTER_CODE = "@399"
SAFETY_DELAY_MINUTES = 10
MOUNTED_OUTPUT_DBFS_URI = "dbfs:/mnt/fsdh-dbk-main-mount/Orion_Only_CURRENT_Trajectory.asc"
WORKSPACE_OUTPUT_PATH = "/Workspace/Users/john.bain@ssc-spc.gc.ca/Artemis 2/raw/Orion_Only_CURRENT_Trajectory.asc"

API_URL = "https://ssd.jpl.nasa.gov/api/horizons.api"

#stop_dt_utc = datetime.now(timezone.utc) - timedelta(minutes=SAFETY_DELAY_MINUTES)
#stop_time_utc = stop_dt_utc.strftime("%Y-%m-%d %H:%M")

#print("Mission start UTC =", MISSION_START_UTC)
#print("Dynamic stop UTC  =", stop_time_utc)
#print("Step size         =", STEP_SIZE)

stop_time_utc = "2026-04-10 23:50"

print("Mission start UTC =", MISSION_START_UTC)
print("Final stop UTC    =", stop_time_utc)
print("Step size         =", STEP_SIZE)


params = {
    "format": "json",
    "COMMAND": f"'{TARGET_COMMAND}'",
    "MAKE_EPHEM": "'YES'",
    "OBJ_DATA": "'NO'",
    "EPHEM_TYPE": "'VECTORS'",
    "CENTER": f"'{CENTER_CODE}'",
    "START_TIME": f"'{MISSION_START_UTC}'",
    "STOP_TIME": f"'{stop_time_utc}'",
    "STEP_SIZE": f"'{STEP_SIZE}'",
    "CSV_FORMAT": "'YES'",
    "OUT_UNITS": "'KM-S'",
    "REF_SYSTEM": "'J2000'",
    "TIME_TYPE": "'UT'",
    "VEC_TABLE": "'2'",
    "VEC_LABELS": "'NO'",
}

resp = requests.get(API_URL, params=params, timeout=60)

print("Request URL:")
print(resp.url)
print("Status code:")
print(resp.status_code)

resp.raise_for_status()

payload = resp.json()

if "error" in payload:
    raise Exception(f"Horizons API error: {payload['error']}")

result_text = payload["result"]

print("Horizons request succeeded.")
print(f"Characters returned: {len(result_text)}")

start_marker = "$$SOE"
end_marker = "$$EOE"

if start_marker not in result_text or end_marker not in result_text:
    raise Exception("Could not find $$SOE / $$EOE markers in Horizons output.")

table_block = result_text.split(start_marker, 1)[1].split(end_marker, 1)[0].strip()

raw_lines = [line.strip() for line in table_block.splitlines() if line.strip()]
print(f"Vector rows found: {len(raw_lines)}")
print("First raw row:")
print(raw_lines[0] if raw_lines else "No rows")

parsed_rows = []

for line in raw_lines:
    fields = [f.strip() for f in next(csv.reader([line], skipinitialspace=True))]

    # Typical Horizons vector row:
    # [jd, calendar_date, x, y, z, vx, vy, vz]
    if len(fields) < 8:
        continue

    if fields[1].startswith("A.D."):
        raw_epoch = fields[1]
        x_idx = 2
    else:
        raw_epoch = fields[0]
        x_idx = 1

    x_km = float(fields[x_idx])
    y_km = float(fields[x_idx + 1])
    z_km = float(fields[x_idx + 2])
    vx_km_s = float(fields[x_idx + 3])
    vy_km_s = float(fields[x_idx + 4])
    vz_km_s = float(fields[x_idx + 5])

    if raw_epoch.startswith("A.D. "):
        epoch_dt = datetime.strptime(raw_epoch, "A.D. %Y-%b-%d %H:%M:%S.%f")
    else:
        epoch_dt = datetime.strptime(raw_epoch, "%Y-%b-%d %H:%M:%S.%f")

    epoch_iso = epoch_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

    parsed_rows.append({
        "epoch_iso": epoch_iso,
        "x_km": x_km,
        "y_km": y_km,
        "z_km": z_km,
        "vx_km_s": vx_km_s,
        "vy_km_s": vy_km_s,
        "vz_km_s": vz_km_s,
    })

if not parsed_rows:
    raise Exception("No parseable state vectors were found.")

print(f"Parsed state vectors: {len(parsed_rows)}")
print(parsed_rows[0])

creation_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
first_epoch = parsed_rows[0]["epoch_iso"]
last_epoch = parsed_rows[-1]["epoch_iso"]

lines = [
    "CCSDS_OEM_VERS = 2.0",
    f"COMMENT Built from JPL Horizons API target {TARGET_COMMAND}",
    f"COMMENT Source center {CENTER_CODE}, units KM-S, ref system J2000",
    f"CREATION_DATE = {creation_date}",
    "ORIGINATOR = FSDH Databricks demo / JPL Horizons",
    "",
    "META_START",
    "OBJECT_NAME = ARTEMIS II",
    f"OBJECT_ID = {TARGET_COMMAND}",
    "CENTER_NAME = EARTH",
    "REF_FRAME = J2000",
    "TIME_SYSTEM = UTC",
    f"START_TIME = {first_epoch}",
    f"USEABLE_START_TIME = {first_epoch}",
    f"USEABLE_STOP_TIME = {last_epoch}",
    f"STOP_TIME = {last_epoch}",
    "META_STOP",
    "",
    "COMMENT State vectors normalized into OEM-like ASC format",
    "",
]

for row in parsed_rows:
    lines.append(
        f"{row['epoch_iso']} "
        f"{row['x_km']:.15f} {row['y_km']:.15f} {row['z_km']:.15f} "
        f"{row['vx_km_s']:.15f} {row['vy_km_s']:.15f} {row['vz_km_s']:.15f}"
    )

asc_text = "\n".join(lines) + "\n"

print("ASC text built successfully.")
print("\n".join(lines[:20]))

dbutils.fs.put(MOUNTED_OUTPUT_DBFS_URI, asc_text, True)
print(f"Saved mounted working copy: {MOUNTED_OUTPUT_DBFS_URI}")

os.makedirs(os.path.dirname(WORKSPACE_OUTPUT_PATH), exist_ok=True)
with open(WORKSPACE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(asc_text)

print(f"Saved workspace mirror copy: {WORKSPACE_OUTPUT_PATH}")

display(dbutils.fs.ls("dbfs:/mnt/fsdh-dbk-main-mount"))

file_text = dbutils.fs.head(MOUNTED_OUTPUT_DBFS_URI, 20000)
preview_lines = file_text.splitlines()[:20]
print("\n".join(preview_lines))